In [276]:
from google.colab import files
uploaded = files.upload()

Saving Clients.csv to Clients.csv


In [277]:
from google.colab import files
uploaded = files.upload()

Saving Passages.csv to Passages.csv


In [278]:
import pandas as pd

clients = pd.read_csv("Clients.csv")
passages = pd.read_csv("Passages.csv")

In [279]:
# vérifier les types
passages.info()
clients.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2197 entries, 0 to 2196
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   ID Client      2197 non-null   int64 
 1   Date Passage   2197 non-null   object
 2   Etablissement  2197 non-null   object
 3   Designation    2197 non-null   object
 4   Quantite       2197 non-null   int64 
 5   Type Forfait   2197 non-null   object
dtypes: int64(2), object(4)
memory usage: 103.1+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   ID Client                  300 non-null    int64 
 1   Date Inscription           300 non-null    object
 2   Date de naissance          300 non-null    object
 3   Etablissement Inscription  300 non-null    object
dtypes: int64(1), object(3)
memory usage: 9.5+ KB


In [280]:
passages.head()

,ID Client,Date Passage,Etablissement,Designation,Quantite,Type Forfait
0,1000,2022-12-29,Arkose Lyon,Abonnement mensuel,1,Abonnement
1,1000,2022-02-09,Arkose Lyon,Pack 10 séances,1,Pack
2,1000,2022-12-21,Arkose Paris,Abonnement mensuel,1,Abonnement
3,1001,2023-09-30,Arkose Paris,Abonnement mensuel,1,Abonnement
4,1001,2023-11-08,Arkose Bordeaux,Abonnement mensuel,1,Abonnement


In [281]:
# vérifier les nulls
passages.isnull().sum()



,0
ID Client,0
Date Passage,0
Etablissement,0
Designation,0
Quantite,0
Type Forfait,0


In [282]:
clients.isnull().sum()

,0
ID Client,0
Date Inscription,0
Date de naissance,0
Etablissement Inscription,0


In [283]:
# vérifier doublons
passages.duplicated().sum()

np.int64(3)

In [ ]:
clients.duplicated().sum()

np.int64(0)

In [290]:
# 2. Convertir les dates
passages["Date Passage"] = pd.to_datetime(passages["Date Passage"], errors="coerce")
clients["Date Inscription"] = pd.to_datetime(clients["Date Inscription"], errors="coerce")
clients["Date de naissance"] = pd.to_datetime(clients["Date de naissance"], errors="coerce")

In [291]:
import sqlite3

conn = sqlite3.connect("arkose.db")

clients.to_sql("clients", conn, index=False, if_exists="replace")
passages.to_sql("passages", conn, index=False, if_exists="replace")

2197

In [292]:
query = """
SELECT
    strftime('%Y', "Date Passage") AS annee,
    strftime('%m', "Date Passage") AS mois,
    SUM(Quantite) AS volume
FROM passages
GROUP BY annee, mois
ORDER BY annee, mois;
"""

df_volume_mois = pd.read_sql(query, conn)
df_volume_mois.head(10)

,annee,mois,volume
0,2022,02,6
1,2022,03,10
2,2022,04,5
3,2022,05,8
4,2022,06,20
5,2022,07,8
6,2022,08,12
7,2022,09,14
8,2022,10,15
9,2022,11,18


In [293]:
query2 = """
SELECT
    "Type Forfait",
    SUM(Quantite) AS volume
FROM passages
GROUP BY "Type Forfait"
ORDER BY volume DESC;
"""

df_volume_forfait = pd.read_sql(query2, conn)
df_volume_forfait

,Type Forfait,volume
0,Abonnement,1231
1,Pack,1086
2,Unité,724


In [294]:
query3 = """
SELECT
    Etablissement,
    SUM(Quantite) AS volume
FROM passages
GROUP BY Etablissement
ORDER BY volume DESC;
"""

df_volume_etab = pd.read_sql(query3, conn)
df_volume_etab

,Etablissement,volume
0,Arkose Paris,1518
1,Arkose Lyon,899
2,Arkose Bordeaux,624


In [296]:
# Dernière visite par client
derniere_visite = passages.groupby("ID Client")["Date Passage"].max().reset_index()
derniere_visite.rename(columns={"Date Passage": "Derniere Visite"}, inplace=True)

#  Date de référence = dernière date du dataset
reference_date = passages["Date Passage"].max()

#  Calcul de la récence
derniere_visite["recence_jours"] = (
    reference_date - derniere_visite["Derniere Visite"]
).dt.days
derniere_visite["recence_jours"].head()

,recence_jours
0,368
1,4
2,59
3,421
4,1


In [298]:
#  Fréquence = nombre de passages par client
frequence = passages.groupby("ID Client").size().reset_index(name="frequence")

#  Volume total = somme des quantités par client
volume = passages.groupby("ID Client")["Quantite"].sum().reset_index(name="volume_total")

#  Fusion des informations
derniere_visite = derniere_visite.merge(frequence, on="ID Client", how="left")
derniere_visite = derniere_visite.merge(volume, on="ID Client", how="left")
derniere_visite.head()

,ID Client,Derniere Visite,recence_jours,frequence_x,volume_total_x,frequence_y,volume_total_y
0,1000,2022-12-29,368,3,3,3,3
1,1001,2023-12-28,4,11,19,11,19
2,1002,2023-11-03,59,9,17,9,17
3,1003,2022-11-06,421,1,1,1,1
4,1004,2023-12-31,1,14,20,14,20


In [303]:
# 9. Segmentation client
def segment_client(recence):
    if recence <= 90:
        return "Très actif"
    elif recence <= 180:
        return "Actif"
    elif recence <= 365:
        return "À risque"
    else:
        return "Inactif"

derniere_visite["segment_client"] = derniere_visite["recence_jours"].apply(segment_client)

# 10. Table finale
segmentation_finale = derniere_visite[
    ["ID Client", "Derniere Visite", "recence_jours", "frequence_x", "volume_total_x", "segment_client"]
].copy()


In [304]:
segmentation_finale = derniere_visite[
    ["ID Client", "Derniere Visite", "recence_jours", "frequence_x", "volume_total_x", "segment_client"]
].copy()

segmentation_finale = segmentation_finale.rename(columns={
    "frequence_x": "frequence",
    "volume_total_x": "volume_total"
})
segmentation_finale.head()

,ID Client,Derniere Visite,recence_jours,frequence,volume_total,segment_client
0,1000,2022-12-29,368,3,3,Inactif
1,1001,2023-12-28,4,11,19,Très actif
2,1002,2023-11-03,59,9,17,Très actif
3,1003,2022-11-06,421,1,1,Inactif
4,1004,2023-12-31,1,14,20,Très actif


In [309]:
from google.colab import files
files.download("segmentation_clients.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>